# Descargando datos climaticos de ERA5Land

El objetivo de este archivo es presentar un tutorial de como descargar datos climaticos de modelos de reanalisis (ERA5Land) desde el Climate Data Store mediante la API disponible utilizando Python. Esta forma facilita las descargas masivas de datos. Este jupyter no corresponde a un tutorial de Python, no representa un tutorial oficial del Climate Data Store y tampoco un tutorial sobre datos climaticos.

Los codigos presentados aqui son para casos de uso basico y se recomienda customizarlos para cumplir con las necesidades de cada proyecto, para ello se puede visitar el [Manual](https://docs.asf.alaska.edu/) para mayor información.

Autor: Javier Norambuena Leiva

## Preparativos

Para descargar datos es *obligatorio* el tener una cuenta en el [ECMWF](https://www.ecmwf.int/) de la unión europea. De esta forma es posible obtener un token de acceso.

Para el token de acceso, uno debe logearse usando las mismas credenciales a la pagina de [Climate Data Store](https://cds.climate.copernicus.eu), luego ir al perfil de usuario y dentro de la pestaña de *your profile*, se encuentra el apartado de API KEY.

Existen dos formas de utilizar esto, la mas segura es usando un archivo de entorno (para linux y macos) o usandolo directamente en el codigo. Se recomienda leer el [manual de instalacion](https://cds.climate.copernicus.eu/how-to-api) para mayor informacion. Para este caso se usara directamente las credenciales en el codigo.

Por otro lado, es necesario instalar la libreria del Climate Data Store la cual posee el cliente que se utiliza para descargar:
- cdsapi

## Construccion de la consulta
Para construir la consulta es necesario definir un par de detalles:
- Años: El producto tiene datos desde 1940 a la fecha, por lo que es necesario definir un rango de años de los que se quieren datos.
- Meses: El producto tiene una la opcion de seleccionar los meses de interes.
- Poligono: Un elemento clave es el poligono de interes, de esta forma la API sabe que pixeles debe tomar al momento de seleccionar y descargar los datos. La estructura del poligo es "[North, West, South, East]" y delimita un cuadrado, los numeros son en grados (90,-90) y (180,-180).
- Variable climatica: El modelo tiene un listado de variables que puede ser llamada, por ejemplo la terperatura superficial del aire ("2m_temperature") entre otras. Por lo tanto, se recomienda revisar las variables disponibles en el [manual del producto ](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview).

En la seccion de ejemplos, puede quedar mas explicito lo anterior.

In [ ]:
import pandas as pd
import os
import time
import warnings
import zipfile
import cdsapi
import zipfile
import numpy as np
import glob

In [ ]:
pol = ##Poligono del area de interes e.g. [70, -170, -90, 180]
anios = ["2000"] ##Lista de años [str] e.g. ["2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
months =["01"] ##Lista de los meses [str] e.g. ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
day = ["01"] ##Lista de los dias a tomar datos e.g. ["01", "02", ...]
variable = ["2m_temperature"] ##Listado con las variables climaticas [str] e.g. ["2m_temperature",]# "10m_u_component_of_wind", "10m_v_component_of_wind",
url = "https://cds.climate.copernicus.eu/api" ##La url es fija
token = ##Token de acceso que esta en el perfil de usuario
path_ERA5_raw = ##Ruta en la que almacenaran los archivos descargados

En esta seccion del codigo se descargan los datos climaticos de interes:

In [ ]:
c = cdsapi.Client(url=url, key=token)
c.retrieve(
    'reanalysis-era5-single-levels', {
        'product_type': ['reanalysis'],
        "variable": variable,
        "year": anios,
        "month": months,
        "day": day,
        "time": [
            "00:00", "01:00", "02:00",
            "03:00", "04:00", "05:00",
            "06:00", "07:00", "08:00",
            "09:00", "10:00", "11:00",
            "12:00", "13:00", "14:00",
            "15:00", "16:00", "17:00",
            "18:00", "19:00", "20:00",
            "21:00", "22:00", "23:00"
        ],
        "data_format": "netcdf",
        "download_format": "zip",
        "area": pol,
    }, path_ERA5_raw+'download.netcdf.zip')
with zipfile.ZipFile(path_ERA5_raw+'download.netcdf.zip', 'r') as zip:
    zip.extractall(path_ERA5_raw)


Por otro lado, aca se descargan los datos referentes al valor geopotencial de la superficie, lo cual permite extraer el valor de altitud al que se calculan las variables (es necesario aplicar una transformacion matematica):

In [ ]:
c.retrieve("reanalysis-era5-single-levels", {
        "product_type": ["reanalysis"],
        "variable": ["geopotential"],
        "year": ["2025"],
        "month": ["06"],
        "day": ["01"],
        "time": ["12:00"],
        "data_format": "netcdf"
    }, path_ERA5_raw+'era5_geopotential_pressure.nc')

## Ejemplos

### Datos de temperatura para Campos de Hielo Norte
Para este ejemplo se descargaran datos de temperatura de todo campos de hielo entre los años 2000 y 2005 de los meses de invierno.

<p align="center">
  <img src="./assets/exploradores.jpg" alt="Glaciar Exploradores photo" width="400"/>
</p>

Foto extraida desde [wikipedia](https://en.wikipedia.org/wiki/Exploradores_Glacier).

In [ ]:
pol = [-46.19, -74.16, -47.7, -72.98]
anios = ["2000", "2001", "2002", "2003", "2004", "2005"]
months =["06", "07", "08"]
day = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31"]
variable = ["2m_temperature"] 
url = "https://cds.climate.copernicus.eu/api" 
token = ##Token de acceso que esta en el perfil de usuario
path_ERA5_raw = "./chn"

c = cdsapi.Client(url=url, key=token)
c.retrieve(
    'reanalysis-era5-single-levels', {
        'product_type': ['reanalysis'],
        "variable": variable,
        "year": anios,
        "month": months,
        "day": day,
        "time": [
            "00:00", "01:00", "02:00",
            "03:00", "04:00", "05:00",
            "06:00", "07:00", "08:00",
            "09:00", "10:00", "11:00",
            "12:00", "13:00", "14:00",
            "15:00", "16:00", "17:00",
            "18:00", "19:00", "20:00",
            "21:00", "22:00", "23:00"
        ],
        "data_format": "netcdf",
        "download_format": "zip",
        "area": pol,
    }, path_ERA5_raw+'download.netcdf.zip')
with zipfile.ZipFile(path_ERA5_raw+'download.netcdf.zip', 'r') as zip:
    zip.extractall(path_ERA5_raw)